第 1 格 · 读取训练表和测试样例

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Set the input and output filenames for the example run.
# 设置样例运行的输入和输出文件名。
train_path = "TrainDataset2026.xls"
test_path = "TestDatasetExample.xls"
output_path = "RFSPrediction_example.csv"

# Read the spreadsheets from the assignment folder.
# 读取作业文件夹里的训练表和测试样例。
train = pd.read_excel(train_path)
test = pd.read_excel(test_path)

第 2 格 · 训练 RFS 岭回归模型

In [2]:
# Keep patients with known RFS labels and remove IDs and outcomes from the features.
# 保留有有效 RFS 标签的患者，从输入特征中去掉 ID 和两个结局列。
target_column = "RelapseFreeSurvival (outcome)"
missing_markers = [999, "999"]

rfs_labels = train[target_column].replace(missing_markers, float("nan"))
has_label = rfs_labels.notna()

X_train = train.loc[has_label].drop(
    columns=["ID", "pCR (outcome)", target_column]
).replace(missing_markers, float("nan"))
y_train = rfs_labels.loc[has_label]

# Treat missing Gene values as a separate category.
# 把 Gene 的缺失值作为一个独立类别，再做独热编码。
gene_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value=-1),
    OneHotEncoder(categories=[[-1, 0, 1]], handle_unknown="ignore"),
)

# Fill these clinical categories with their training-set modes.
# 用训练集中各列的众数填补这些临床类别特征。
mode_columns = [
    "ER", "HER2", "PgR", "TrippleNegative", "ChemoGrade",
    "HistologyType", "LNStatus", "Proliferation",
]

# Use the training-set median for the remaining numeric features.
# 其余数值特征用训练集中各列的中位数填补。
preprocessing = ColumnTransformer(
    [
        ("gene", gene_pipeline, ["Gene"]),
        ("clinical_mode", SimpleImputer(strategy="most_frequent"), mode_columns),
    ],
    remainder=SimpleImputer(strategy="median"),
    sparse_threshold=0,
)

# Fit the full-feature ridge model with the same alpha as the development notebook.
# 沿用开发本的全特征岭回归配置，使用 alpha=1000 训练。
model = make_pipeline(preprocessing, StandardScaler(), Ridge(alpha=1000))
model.fit(X_train, y_train)
print(f"Training patients: {len(y_train)}")

Training patients: 400


第 3 格 · 预测并导出 CSV

In [3]:
# Select the same feature columns in the same order as in training.
# 按训练时的列名和顺序提取测试特征，并转换缺失标记。
X_test = test.loc[:, X_train.columns].replace(missing_markers, float("nan"))

# Write the patient ID first and the RFS prediction second.
# 第一列写患者 ID，第二列写 RFS 预测值，保持测试表的行顺序。
predictions = pd.DataFrame({
    "ID": test["ID"],
    "RFS": model.predict(X_test),
})
predictions.to_csv(output_path, index=False)
display(predictions.head())

,ID,RFS
0,TRG002728,53.260571
1,TRG002649,52.442652
2,TRG002628,47.200299


点 Restart 重启内核，然后点 全部运行。第 2 格应显示 400 条训练记录，第 3 格应显示 3 行、2 列，RFS 预测依次约为 53.26、52.44、47.20。这次会覆盖现有的样例 CSV。

正式测试集发布后，只需要在第 1 格把 test_path 改成正式测试表的文件名，把 output_path 改成 RFSPrediction.csv，再点一次 全部运行。